# Reviewer 2 training loop for Google Colab

This notebook is a crash-resilient Colab runner for the Reviewer 2 experiment. It keeps datasets, checkpoints, normalization, split manifests, per-epoch state, and results on Google Drive.

Runtime profiles:
- `quick`: one seed, five epochs, smoke test only;
- `pilot`: two seeds, ten epochs; useful for checking the pipeline;
- `full`: five seeds, 30 epochs, plus eight fixed LUDB adaptation epochs for R6.

The quick and pilot profiles are not paper results. Only the full profile satisfies the repeated-seed Reviewer 2 protocol.

In [ ]:
# Colab setup: install dependencies, mount Drive, and create persistent paths.
%pip -q install wfdb scipy scikit-learn pandas numpy torch

from pathlib import Path
import json, os, random, shutil, time
import numpy as np
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/PQRST_Reviewer2')
DATA_ROOT = DRIVE_ROOT / 'datasets'
QTDB_DIR = DATA_ROOT / 'qtdb-1.0.0'
LUDB_DIR = DATA_ROOT / 'ludb-1.0.1'
ARTIFACT_DIR = DRIVE_ROOT / 'reviewer2_artifacts'
CHECKPOINT_DIR = ARTIFACT_DIR / 'checkpoints'
STATE_DIR = ARTIFACT_DIR / 'state'
for folder in [DATA_ROOT, ARTIFACT_DIR, CHECKPOINT_DIR, STATE_DIR]: folder.mkdir(parents=True, exist_ok=True)

PROFILE = 'quick'  # change to 'pilot' or 'full' before running the training cell
PROFILES = {
    'quick': {'seeds': [1], 'epochs': 5},
    'pilot': {'seeds': [1, 2], 'epochs': 10},
    'full': {'seeds': [1, 2, 3, 4, 5], 'epochs': 30},
}
if PROFILE not in PROFILES: raise ValueError(PROFILE)
TRAINING_SEEDS = PROFILES[PROFILE]['seeds']
NUM_EPOCHS = PROFILES[PROFILE]['epochs']
PRE, POST, R5_POST = 120, 240, 320
BATCH_SIZE = 64
R6_ADAPT_EPOCHS = 8
DEVICE = __import__('torch').device('cuda' if __import__('torch').cuda.is_available() else 'cpu')
print('PROFILE:', PROFILE, '| seeds:', TRAINING_SEEDS, '| epochs:', NUM_EPOCHS, '| device:', DEVICE)

## Dataset sources and persistence

Official sources:
- QTDB: https://physionet.org/content/qtdb/1.0.0/
- LUDB: https://physionet.org/content/ludb/1.0.1/

The download cell uses WFDB and writes directly to Drive. It skips a dataset when paired `.hea` and signal files already exist, so a new Colab session does not redownload data.

In [ ]:
import wfdb

def paired_records(folder):
    headers = {path.stem for path in Path(folder).glob('*.hea')}
    signals = {path.stem for path in Path(folder).glob('*.dat')}
    return sorted(headers & signals)

def download_once(database, target, expected_count):
    target = Path(target); target.mkdir(parents=True, exist_ok=True)
    complete = target / '.download_complete.json'
    records = paired_records(target)
    if complete.exists() and len(records) == expected_count:
        print(database, 'already present:', len(records), 'records')
        return records
    print('Downloading', database, 'to', target, '- this is persisted on Drive')
    wfdb.dl_database(database, dl_dir=str(target), keep_subdirs=False)
    records = paired_records(target)
    if len(records) != expected_count:
        raise RuntimeError(f'{database}: expected {expected_count} paired records, found {len(records)}')
    complete.write_text(json.dumps({'database': database, 'records': records, 'time': time.time()}, indent=2))
    return records

qtdb_records = download_once('qtdb', QTDB_DIR, 105)
ludb_records = download_once('ludb', LUDB_DIR, 200)
print('QTDB:', len(qtdb_records), '| LUDB:', len(ludb_records))

## Persistence and crash recovery

Every write goes through a temporary file and atomic rename. The epoch checkpoint contains model weights, optimizer state, best validation state, current epoch, seed, profile, and provenance. If Colab disconnects, remount Drive, rerun setup/import cells, and rerun the training cell; completed epochs and seeds are skipped or resumed.

In [ ]:
import tempfile
import torch

def atomic_torch_save(value, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=path.parent, suffix='.tmp', delete=False) as handle:
        temporary = Path(handle.name)
    try:
        torch.save(value, temporary)
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)

def atomic_json_save(value, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, default=lambda item: item.item() if isinstance(item, np.generic) else str(item)))
    temporary.replace(path)

def checkpoint_path(seed, name, epoch=None):
    suffix = 'latest' if epoch is None else f'epoch_{epoch:02d}'
    return CHECKPOINT_DIR / f'{PROFILE}_seed_{seed}_{name}_{suffix}.pth'

def save_npz_atomic(path, **arrays):
    path = Path(path); temporary = path.with_suffix('.tmp.npz')
    np.savez(temporary, **arrays)
    temporary.replace(path)

def save_seed_progress(seed, payload):
    atomic_json_save(payload, STATE_DIR / f'{PROFILE}_seed_{seed}_progress.json')

print('Persistent artifact root:', ARTIFACT_DIR)

In [ ]:
# Record-level splits and train-only normalization. This cell is intentionally resumable.
from sklearn.model_selection import train_test_split
from scipy.signal import butter, sosfiltfilt, find_peaks, resample_poly
import urllib.request

REVIEWER_NOTEBOOK = DRIVE_ROOT / 'reviewer2_matched_metrics.ipynb'
REVIEWER_URL = 'https://raw.githubusercontent.com/vrishank-na/PQRST_mapping/main/reviewer2_matched_metrics.ipynb'
if not REVIEWER_NOTEBOOK.exists():
    print('Downloading reviewer source notebook to Drive...')
    urllib.request.urlretrieve(REVIEWER_URL, REVIEWER_NOTEBOOK)
reviewer_nb = json.loads(REVIEWER_NOTEBOOK.read_text())

# Load only shared helper definitions; do not execute stale outputs or training.
helper_source = ''.join(reviewer_nb['cells'][3]['source'])
exec(helper_source, globals())

qtdb_train_records, qtdb_val_records = train_test_split(qtdb_records, test_size=0.20, random_state=7, shuffle=True)
qtdb_train_records, qtdb_val_records = sorted(qtdb_train_records), sorted(qtdb_val_records)
r6_adapt_records, r6_test_records = train_test_split(ludb_records, test_size=0.90, random_state=17, shuffle=True)
r6_adapt_records, r6_test_records = sorted(r6_adapt_records), sorted(r6_test_records)
assert len(qtdb_train_records) == 84 and len(qtdb_val_records) == 21
assert len(r6_adapt_records) == 20 and len(r6_test_records) == 180
assert set(qtdb_train_records).isdisjoint(qtdb_val_records)
assert set(r6_adapt_records).isdisjoint(r6_test_records)

# Execute the reviewed partition builder so windows are created only after record splits.
partition_source = ''.join(reviewer_nb['cells'][5]['source'])
exec(partition_source, globals())
manifest = {'profile': PROFILE, 'training_seeds': TRAINING_SEEDS, 'epochs': NUM_EPOCHS, 'qtdb_directory': str(QTDB_DIR), 'ludb_directory': str(LUDB_DIR), 'qtdb_train_records': qtdb_train_records, 'qtdb_validation_records': qtdb_val_records, 'ludb_adaptation_records': r6_adapt_records, 'ludb_test_records': r6_test_records, 'split_seeds': {'qtdb': 7, 'ludb': 17}, 'r6_adaptation_epochs': R6_ADAPT_EPOCHS}
manifest.update({
    'qtdb_train_windows_240': len(X_qt_train),
    'qtdb_validation_windows_240': len(X_qt_val),
    'ludb_adaptation_windows_240': len(X_lu_adapt_240),
    'ludb_test_windows_240': len(X_lu_test_240),
    'qtdb_train_mean_240': train_mean,
    'qtdb_train_std_240': train_std,
})
atomic_json_save(manifest, ARTIFACT_DIR / 'colab_run_manifest.json')
np.savez(ARTIFACT_DIR / 'reviewer2_qtdb_normalization.npz', mean=train_mean, std=train_std)
print('Preprocessing and manifest persisted:', ARTIFACT_DIR / 'colab_run_manifest.json')

## Resumable training harness

This is the control layer to place around the existing reviewer model/preprocessing functions. It checkpoints after every epoch, writes a progress manifest after every epoch, and writes a completed-seed result immediately. For a full run, use the model/training function definitions from the reviewer notebook in the next cell or paste them into this persistent notebook.

The guard below refuses to start the expensive loop until the required functions are defined, avoiding an accidental partial run.

In [ ]:
# Load reviewed model/training definitions automatically from the persisted source notebook.
model_source = next(''.join(cell['source']) for cell in reviewer_nb['cells'] if any('class CNNFeatureExtractor' in line for line in cell['source']))
exec(model_source.split("# R4 threshold is selected", 1)[0], globals())
training_source = next(''.join(cell['source']) for cell in reviewer_nb['cells'] if any('def train_fixed_model' in line for line in cell['source']))
exec(training_source.split('seed_rows, seed_provenance', 1)[0], globals())

REQUIRED_FUNCTIONS = ['RPeakGuidedML2', 'RPeakTimeML2', 'FocalLoss', 'make_loader', 'run_training_epoch', 'predict', 'time_channel']
missing = [name for name in REQUIRED_FUNCTIONS if name not in globals()]
if missing: raise RuntimeError(f'Reviewer definitions failed to load: {missing}')


def resumable_train(model, train_features, train_labels, val_features, val_labels, criterion, seed, name):
    set_global_seed(seed)
    train_loader = make_loader(train_features, train_labels, seed, shuffle=True)
    val_loader = make_loader(val_features, val_labels, seed, shuffle=False)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    latest_path = checkpoint_path(seed, name)
    start_epoch, best_loss, best_epoch, best_state = 0, float('inf'), 0, None
    if latest_path.exists():
        state = torch.load(latest_path, map_location=DEVICE)
        model.load_state_dict(state['model_state_dict'])
        optimizer.load_state_dict(state['optimizer_state_dict'])
        start_epoch = int(state.get('epoch', 0))
        best_loss = float(state.get('best_validation_loss', float('inf')))
        best_epoch = int(state.get('best_epoch', 0))
        best_state = state.get('best_model_state_dict')
        print(f'Resuming {name}, seed {seed}, after epoch {start_epoch}')
    for epoch in range(start_epoch, NUM_EPOCHS):
        train_loss = run_training_epoch(model, train_loader, criterion, optimizer)
        val_loss = run_training_epoch(model, val_loader, criterion)
        if val_loss < best_loss:
            best_loss, best_epoch = val_loss, epoch + 1
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        payload = {'epoch': epoch + 1, 'best_validation_loss': best_loss, 'best_epoch': best_epoch, 'seed': seed, 'name': name, 'profile': PROFILE, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'best_model_state_dict': best_state}
        atomic_torch_save(payload, latest_path)
        atomic_torch_save(payload, checkpoint_path(seed, name, epoch + 1))
        save_seed_progress(seed, {'last_completed_model': name, 'last_completed_epoch': epoch + 1, 'best_epoch': best_epoch, 'best_validation_loss': best_loss, 'profile': PROFILE, 'time': time.time()})
        print(f'{name} seed={seed} epoch={epoch + 1}/{NUM_EPOCHS}: train={train_loss:.4f} val={val_loss:.4f} persisted')
    if best_state is not None: model.load_state_dict(best_state)
    return model, {'best_epoch': best_epoch, 'best_validation_loss': best_loss}


def completed_seed_path(seed):
    return STATE_DIR / f'{PROFILE}_seed_{seed}_complete.json'


def save_completed_seed(seed, rows, provenance):
    atomic_json_save({'seed': seed, 'metrics': rows, 'provenance': provenance, 'completed_at': time.time()}, completed_seed_path(seed))
    pd.DataFrame(rows).to_csv(ARTIFACT_DIR / f'{PROFILE}_seed_{seed}_metrics.csv', index=False)

print('Definitions loaded; resumable training is ready.')

In [ ]:
seed_rows = []

for seed in TRAINING_SEEDS:
    completed = completed_seed_path(seed)
    if completed.exists():
        saved = json.loads(completed.read_text())
        seed_rows.extend(saved['metrics'])
        print('Seed already complete; loaded persisted results:', seed)
        continue

    seed_provenance = {'seed': seed, 'profile': PROFILE, 'qtdb_train_records': qtdb_train_records, 'qtdb_validation_records': qtdb_val_records, 'ludb_adaptation_records': r6_adapt_records, 'ludb_test_records': r6_test_records, 'qtdb_train_mean_240': train_mean, 'qtdb_train_std_240': train_std, 'r6_adaptation_epochs': R6_ADAPT_EPOCHS}
    rows = []

    r1, r1_info = resumable_train(RPeakGuidedML2().to(DEVICE), X_qt_train[:, None], Y_qt_train, X_qt_val[:, None], Y_qt_val, nn.CrossEntropyLoss(), seed, 'R1')
    r1_info['post_samples'] = POST
    rows.append(classification_row('R1', Y_lu_test_240, predict(r1, X_lu_test_240[:, None]), lu_test_ids_240, POST) | {'Seed': seed, **r1_info})

    class_counts = np.bincount(Y_qt_train.ravel(), minlength=3).astype(np.float32)
    weights = class_counts.sum() / np.maximum(class_counts, 1.0)
    weights = torch.tensor(weights / weights.mean(), dtype=torch.float32, device=DEVICE)
    r2, r2_info = resumable_train(RPeakGuidedML2().to(DEVICE), X_qt_train[:, None], Y_qt_train, X_qt_val[:, None], Y_qt_val, FocalLoss(alpha=weights), seed, 'R2')
    r2_info['post_samples'] = POST
    rows.append(classification_row('R2', Y_lu_test_240, predict(r2, X_lu_test_240[:, None]), lu_test_ids_240, POST) | {'Seed': seed, **r2_info})

    r3, r3_info = resumable_train(RPeakGuidedML2().to(DEVICE), X_qt_train[:, None], Y_qt_train, X_qt_val[:, None], Y_qt_val, FocalLoss(), seed, 'R3')
    r3_info['post_samples'] = POST
    rows.append(classification_row('R3', Y_lu_test_240, predict(r3, X_lu_test_240[:, None]), lu_test_ids_240, POST) | {'Seed': seed, **r3_info})

    r4_val_probabilities = predict_probabilities(r1, X_qt_val[:, None])
    threshold_grid = []
    for threshold in np.arange(0.10, 0.76, 0.05):
        decoded = r4_decode(r4_val_probabilities, threshold)
        threshold_grid.append((threshold, f1_score(Y_qt_val.ravel(), decoded.ravel(), labels=[0, 1, 2], average='macro', zero_division=0)))
    r4_threshold = float(max(threshold_grid, key=lambda item: item[1])[0])
    r4_prediction = r4_decode(predict_probabilities(r1, X_lu_test_240[:, None]), r4_threshold)
    rows.append(classification_row('R4', Y_lu_test_240, r4_prediction, lu_test_ids_240, POST) | {'Seed': seed, 'Best epoch': r1_info['best_epoch'], 'Best validation loss': r1_info['best_validation_loss'], 'R4 threshold': r4_threshold})

    r5, r5_info = resumable_train(RPeakGuidedML2().to(DEVICE), X_qt_train_320[:, None], Y_qt_train_320, X_qt_val_320[:, None], Y_qt_val_320, nn.CrossEntropyLoss(), seed, 'R5')
    r5_info['post_samples'] = R5_POST
    rows.append(classification_row('R5', Y_lu_test_320, predict(r5, X_lu_test_320[:, None]), lu_test_ids_320, R5_POST) | {'Seed': seed, **r5_info})

    r6, r6_info = resumable_train(RPeakTimeML2().to(DEVICE), time_channel(X_qt_train_320, R5_POST), Y_qt_train_320, time_channel(X_qt_val_320, R5_POST), Y_qt_val_320, nn.CrossEntropyLoss(), seed, 'R6_QTDB')
    adaptation_loader = make_loader(time_channel(X_lu_adapt_320, R5_POST), Y_lu_adapt_320, seed, shuffle=True)
    adaptation_optimizer = torch.optim.Adam(r6.parameters(), lr=1e-4)
    adaptation_state_path = checkpoint_path(seed, 'R6_adaptation')
    adaptation_start = 0
    if adaptation_state_path.exists():
        adaptation_state = torch.load(adaptation_state_path, map_location=DEVICE)
        r6.load_state_dict(adaptation_state['model_state_dict'])
        adaptation_optimizer.load_state_dict(adaptation_state['optimizer_state_dict'])
        adaptation_start = int(adaptation_state.get('epoch', 0))
    for adaptation_epoch in range(adaptation_start, R6_ADAPT_EPOCHS):
        adaptation_loss = run_training_epoch(r6, adaptation_loader, nn.CrossEntropyLoss(), adaptation_optimizer)
        atomic_torch_save({'epoch': adaptation_epoch + 1, 'seed': seed, 'name': 'R6_adaptation', 'model_state_dict': r6.state_dict(), 'optimizer_state_dict': adaptation_optimizer.state_dict(), 'adaptation_loss': adaptation_loss}, adaptation_state_path)
        print(f'R6 adaptation seed={seed} epoch={adaptation_epoch + 1}/{R6_ADAPT_EPOCHS}: loss={adaptation_loss:.4f} persisted')
    rows.append(classification_row('R6 adapted', Y_lu_test_320, predict(r6, time_channel(X_lu_test_320, R5_POST)), lu_test_ids_320, R5_POST) | {'Seed': seed, 'Best epoch': r6_info['best_epoch'], 'Best validation loss': r6_info['best_validation_loss'], 'R6 adaptation epochs': R6_ADAPT_EPOCHS})

    seed_rows.extend(rows)
    save_completed_seed(seed, rows, seed_provenance)
    pd.DataFrame(seed_rows).to_csv(ARTIFACT_DIR / f'{PROFILE}_metrics_so_far.csv', index=False)
    print('Completed and persisted seed:', seed)

seed_metrics = pd.DataFrame(seed_rows)
seed_metrics.to_csv(ARTIFACT_DIR / f'{PROFILE}_metrics.csv', index=False)
display(seed_metrics.round(4))

In [ ]:
from scipy.stats import t as student_t


def mean_sd_ci(values):
    values = np.asarray(values, dtype=float)
    mean = float(values.mean())
    sd = float(values.std(ddof=1)) if len(values) > 1 else 0.0
    critical = float(student_t.ppf(0.975, len(values) - 1)) if len(values) > 1 else np.nan
    margin = critical * sd / np.sqrt(len(values)) if len(values) > 1 else np.nan
    return mean, sd, mean - margin if len(values) > 1 else np.nan, mean + margin if len(values) > 1 else np.nan

summary_rows = []
for experiment, group in seed_metrics.groupby('Experiment'):
    for metric in ['P F1', 'T F1', 'Macro F1', 'Weighted F1', 'Accuracy']:
        mean, sd, low, high = mean_sd_ci(group[metric])
        summary_rows.append({'Experiment': experiment, 'Metric': metric, 'Seeds': len(group), 'Mean': mean, 'SD': sd, '95% CI low': low, '95% CI high': high})
summary = pd.DataFrame(summary_rows)
summary.to_csv(ARTIFACT_DIR / f'{PROFILE}_mean_sd_t_ci.csv', index=False)

pivot = seed_metrics.pivot(index='Seed', columns='Experiment', values='Macro F1')
if {'R3', 'R6 adapted'} <= set(pivot.columns):
    differences = pivot['R6 adapted'] - pivot['R3']
    mean, sd, low, high = mean_sd_ci(differences)
    difference_table = pd.DataFrame({'Seed': differences.index, 'R3 Macro F1': pivot['R3'], 'R6 Macro F1': pivot['R6 adapted'], 'R6 minus R3': differences})
    difference_summary = pd.DataFrame([{'Comparison': 'R6 adapted - R3', 'Mean': mean, 'SD': sd, '95% CI low': low, '95% CI high': high}])
    difference_table.to_csv(ARTIFACT_DIR / f'{PROFILE}_r3_r6_by_seed.csv', index=False)
    difference_summary.to_csv(ARTIFACT_DIR / f'{PROFILE}_r3_r6_difference_ci.csv', index=False)
    display(difference_table.round(4)); display(difference_summary.round(4))

display(summary.round(4))
print('All summaries persisted to:', ARTIFACT_DIR)

In [ ]:
from datetime import datetime, timezone

reviewer2_checks = {
    'QTDB complete population': len(qtdb_records) == 105,
    'QTDB split 84/21': len(qtdb_train_records) == 84 and len(qtdb_val_records) == 21,
    'LUDB complete population': len(ludb_records) == 200,
    'LUDB split 20/180': len(r6_adapt_records) == 20 and len(r6_test_records) == 180,
    'QTDB record disjointness': set(qtdb_train_records).isdisjoint(qtdb_val_records),
    'LUDB record disjointness': set(r6_adapt_records).isdisjoint(r6_test_records),
    'train-only normalization persisted': (ARTIFACT_DIR / 'reviewer2_qtdb_normalization.npz').exists(),
    'split manifest persisted': (ARTIFACT_DIR / 'colab_run_manifest.json').exists(),
    'all five seeds configured': TRAINING_SEEDS == [1, 2, 3, 4, 5],
    'full 30-epoch profile selected': PROFILE == 'full' and NUM_EPOCHS == 30,
}
completed_seeds = sorted(int(path.stem.split('_seed_')[1].split('_')[0]) for path in STATE_DIR.glob(f'{PROFILE}_seed_*_complete.json'))
reviewer2_checks['five seeds completed'] = completed_seeds == [1, 2, 3, 4, 5]
final_reviewer2_results = {
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'status': 'PASS' if all(reviewer2_checks.values()) else 'INCOMPLETE',
    'checks': reviewer2_checks,
    'profile': PROFILE,
    'completed_seeds': completed_seeds,
    'dataset_counts': {'qtdb_total': len(qtdb_records), 'qtdb_train': len(qtdb_train_records), 'qtdb_validation': len(qtdb_val_records), 'ludb_total': len(ludb_records), 'ludb_adaptation': len(r6_adapt_records), 'ludb_test': len(r6_test_records)},
    'artifacts': {'manifest': str(ARTIFACT_DIR / 'colab_run_manifest.json'), 'normalization': str(ARTIFACT_DIR / 'reviewer2_qtdb_normalization.npz'), 'checkpoint_directory': str(CHECKPOINT_DIR), 'state_directory': str(STATE_DIR), 'metrics': str(ARTIFACT_DIR / f'{PROFILE}_metrics.csv')},
    'limitations': ['Quick and pilot profiles are validation runs, not Reviewer 2 final results.', 'The current target representation is background/P/T; QRS delineation is not claimed.', 'No event-boundary or external-delineator result is included in this Reviewer 2 package.'],
}
atomic_json_save(final_reviewer2_results, ARTIFACT_DIR / 'FINAL_REVIEWER_2_RESULTS.json')
for name, passed in reviewer2_checks.items(): print(f'{"PASS" if passed else "FAIL"}: {name}')
print('FINAL_REVIEWER_2_RESULTS:', final_reviewer2_results['status'])

## Recommended cloud runtimes

- Cheapest smoke test: Colab free T4/L4 if available, `PROFILE='quick'`.
- Good cost/performance: Colab Pro with T4 or L4, `PROFILE='pilot'` first, then `full`.
- Faster but more expensive: A100, use only if the five-seed deadline matters.
- Avoid CPU runtimes for this experiment.

Practical workflow:
1. Run setup, download, persistence, and split cells once.
2. Run one quick seed and confirm checkpoints appear in Drive.
3. Switch to `pilot` or `full`.
4. After each seed, verify `reviewer2_artifacts/state/` and `checkpoints/` in Drive.
5. If Colab crashes, reconnect, remount Drive, rerun setup/import cells, and resume from the latest checkpoint.

Drive persistence protects files, but it cannot recover a training step that was never checkpointed. Keep checkpoint frequency at every epoch.